# Dashboard 7 · rubric additions
Adds the before/after-cleaning plot and the numbers for the conclusion. After each cell, copy the printed sentence under the plot on your slide.

In [3]:
# Dashboard 7 · rubric additions (before/after cleaning + numbers for the conclusion)
# Run from the same folder as your other notebooks. Paths: edit RAW_P / CLEAN_P / JOIN_P if yours differ.
import numpy as np, pandas as pd, plotly.graph_objects as go
from plotly.subplots import make_subplots
RAW_P   = "interim/nav/nav_midcap_combined.parquet"
CLEAN_P = "clean/nav_midcap_clean.parquet"
JOIN_P  = "clean/nav_ter_midcap_joined.parquet"
GREY, HI, DARK = "#9aa0a6", "#1f4e9c", "#222222"   # grey = before, one highlight colour = after (readable in greyscale)


## Plot A · before vs after cleaning

In [4]:
# ---- Plot A: daily returns BEFORE vs AFTER cleaning ----
raw = pd.read_parquet(RAW_P)
raw = raw[raw["Option"].str.upper().str.contains("GROWTH")]
raw["NAV"]  = pd.to_numeric(raw["Net Asset Value"], errors="coerce")
raw["Date"] = pd.to_datetime(raw["Date"], format="%d-%b-%Y", errors="coerce")
raw = raw.sort_values(["Scheme Code", "Date"])
gap = raw.groupby("Scheme Code")["Date"].diff().dt.days
raw["ret"] = raw.groupby("Scheme Code")["NAV"].pct_change() * 100
raw.loc[gap > 7, "ret"] = np.nan

cl = pd.read_parquet(CLEAN_P).sort_values(["MF_ID", "Plan", "Date"])
gap = cl.groupby(["MF_ID", "Plan"])["Date"].diff().dt.days
cl["ret"] = cl.groupby(["MF_ID", "Plan"])["NAV"].pct_change() * 100
cl.loc[gap > 7, "ret"] = np.nan

b, a = raw["ret"].dropna(), cl["ret"].dropna()
nb_, na_ = int((b.abs() > 3).sum()), int((a.abs() > 3).sum())
bins = dict(start=-20, end=20, size=0.5)
fig = go.Figure()
fig.add_histogram(x=b.clip(-20, 20), xbins=bins, name=f"Before cleaning (n={len(b):,})",
                  marker=dict(color="rgba(0,0,0,0)", line=dict(color=GREY, width=1.5)))
fig.add_histogram(x=a.clip(-20, 20), xbins=bins, name=f"After cleaning (n={len(a):,})",
                  marker=dict(color=HI, opacity=0.75))
fig.update_yaxes(type="log", title_text="Number of fund-days (log scale)")
fig.update_xaxes(title_text="Daily NAV return (%), clipped at ±20")
fig.update_layout(barmode="overlay", title="Daily NAV returns before vs after cleaning",
                  template="simple_white", legend=dict(orientation="h", y=1.1), height=420)
fig.show()
print(f"Days with |return| > 3%: {nb_:,} before vs {na_:,} after cleaning "
      f"(largest move {b.abs().max():.1f}% before, {a.abs().max():.1f}% after).")


Days with |return| > 3%: 1,644 before vs 1,623 after cleaning (largest move 13.9% before, 13.9% after).


## Numbers for the conclusion

In [5]:
# ---- Numbers for the conclusion: TER gap and TER vs return, from the joined table ----
j = pd.read_parquet(JOIN_P).dropna(subset=["NAV", "TER"])
YEARS = 3
end = j["Date"].max(); start = end - pd.DateOffset(years=YEARS)

# 1) direct vs regular TER gap (latest TER per house and plan)
last = j.sort_values("Date").groupby(["MF_ID", "Plan"])["TER"].last().unstack()
gap = (last["Regular Plan"] - last["Direct Plan"]).dropna()
print(f"TER gap (Regular - Direct): mean {gap.mean():.2f} pp, range {gap.min():.2f} to {gap.max():.2f} pp across {len(gap)} fund houses.")

# 2) TER vs CAGR over the last YEARS years, per plan
from scipy import stats
rows = []
for plan, g in j.groupby("Plan"):
    w = g.pivot_table(index="Date", columns="MF_ID", values="NAV")
    s0 = w[w.index <= start].ffill().iloc[-1] if (w.index <= start).any() else None
    if s0 is None: continue
    s1 = w.ffill().iloc[-1]
    cagr = ((s1 / s0) ** (1 / YEARS) - 1) * 100
    ter = g[g["Date"] >= start].groupby("MF_ID")["TER"].mean()
    d = pd.concat([ter.rename("ter"), cagr.rename("cagr")], axis=1).dropna()
    r, p = stats.pearsonr(d["ter"], d["cagr"]); slope = np.polyfit(d["ter"], d["cagr"], 1)[0]
    rows.append(dict(plan=plan, houses=len(d), mean_ter=d["ter"].mean(), mean_cagr=d["cagr"].mean(), r=r, p=p, slope=slope))
res = pd.DataFrame(rows).round(3); display(res)
for _, x in res.iterrows():
    print(f"{x['plan']}: across {int(x['houses'])} houses, r = {x['r']:.2f} (p = {x['p']:.2f}); each +1 pp of TER goes with {x['slope']:+.1f} pp of {YEARS}-year CAGR.")
if len(res) == 2:
    dd = res.set_index("plan")
    print(f"Average {YEARS}-year CAGR: Direct {dd.loc['Direct Plan','mean_cagr']:.1f}% vs Regular {dd.loc['Regular Plan','mean_cagr']:.1f}% "
          f"= {dd.loc['Direct Plan','mean_cagr']-dd.loc['Regular Plan','mean_cagr']:.1f} pp a year.")


TER gap (Regular - Direct): mean 1.04 pp, range -0.54 to 1.63 pp across 33 fund houses.


,plan,houses,mean_ter,mean_cagr,r,p,slope
0,Direct Plan,28,0.790,15.496,-0.388,0.041,-3.886
1,Regular Plan,28,1.892,14.224,-0.054,0.787,-0.665


Direct Plan: across 28 houses, r = -0.39 (p = 0.04); each +1 pp of TER goes with -3.9 pp of 3-year CAGR.
Regular Plan: across 28 houses, r = -0.05 (p = 0.79); each +1 pp of TER goes with -0.7 pp of 3-year CAGR.
Average 3-year CAGR: Direct 15.5% vs Regular 14.2% = 1.3 pp a year.
